# Transfer Learning — Fine-Tuning Pre-Trained Models

Why train from scratch when you can leverage **millions of images** already learned?

- **What is Transfer Learning?** — Re-using pre-trained model weights
- **Feature Extraction**: Freeze pre-trained layers, train new classifier
- **Fine-Tuning**: Unfreeze some layers and retrain
- **Models**: VGG16, ResNet50, MobileNetV2
- **Practical**: CIFAR-10 classification with transfer learning

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import warnings
warnings.filterwarnings('ignore')
print(f'TensorFlow: {tf.__version__}')

In [ ]:
# Load CIFAR-10
(X_train, y_train), (X_test, y_test) = cifar10.load_data()
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# Resize to 32x32 → We'll upsample for pretrained models (they expect 224x224 min)
# For efficiency, we'll use a subset
X_train_sub = X_train[:5000]
y_train_sub = y_train_cat[:5000]
print(f'Train subset: {X_train_sub.shape} | Test: {X_test.shape}')

## 1. Transfer Learning Concepts

```
Pre-trained Model (ImageNet)         Your Task
┌─────────────────────────┐         ┌──────────────┐
│ Conv1 → Conv2 → ... →  │ ──────→ │ Dense → Out  │
│  (feature extractor)    │ frozen  │ (new head)   │
│  Learned on 1M images   │         │ Trained on   │
└─────────────────────────┘         │ your data    │
                                    └──────────────┘
```

**Strategy 1 — Feature Extraction**: Freeze ALL pre-trained layers  
**Strategy 2 — Fine-Tuning**: Unfreeze top layers and retrain with low LR

## 2. Available Pre-Trained Models

In [ ]:
# Compare architectures
import pandas as pd
models_info = pd.DataFrame({
    'Model': ['VGG16', 'ResNet50', 'MobileNetV2', 'InceptionV3', 'EfficientNetB0'],
    'Parameters (M)': [138.4, 25.6, 3.4, 23.9, 5.3],
    'Top-1 Acc (ImageNet)': [71.3, 74.9, 71.3, 77.9, 77.1],
    'Size (MB)': [528, 98, 14, 92, 29],
    'Best For': ['Teaching', 'General', 'Mobile/Edge', 'High accuracy', 'Best tradeoff']
})
print(models_info.to_string(index=False))

## 3. Feature Extraction with MobileNetV2

In [ ]:
# Strategy 1: Feature Extraction (freeze all pre-trained layers)
def build_feature_extractor(base_model_class, input_shape=(32, 32, 3), num_classes=10):
    # Load pre-trained model without top classification layers
    base_model = base_model_class(weights='imagenet', include_top=False, 
                                  input_shape=input_shape)
    base_model.trainable = False  # Freeze all layers
    
    # Add custom classifier
    inputs = keras.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model, base_model

fe_model, base = build_feature_extractor(MobileNetV2)
print(f'Total params: {fe_model.count_params():,}')
print(f'Trainable params: {sum(tf.keras.backend.count_params(w) for w in fe_model.trainable_weights):,}')
print(f'Non-trainable params: {sum(tf.keras.backend.count_params(w) for w in fe_model.non_trainable_weights):,}')

In [ ]:
# Train feature extractor
early_stop = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

h_fe = fe_model.fit(X_train_sub, y_train_sub, epochs=20, batch_size=32,
                     validation_data=(X_test[:2000], y_test_cat[:2000]),
                     callbacks=[early_stop], verbose=1)

fe_score = fe_model.evaluate(X_test, y_test_cat, verbose=0)
print(f'\nFeature Extraction — Test Accuracy: {fe_score[1]:.4f}')

## 4. Fine-Tuning — Unfreeze Top Layers

In [ ]:
# Strategy 2: Fine-tuning — unfreeze last N layers
base.trainable = True

# Freeze everything except last 20 layers
for layer in base.layers[:-20]:
    layer.trainable = False

trainable_count = sum(1 for l in base.layers if l.trainable)
print(f'Fine-tuning: {trainable_count} / {len(base.layers)} layers trainable')

# IMPORTANT: Use a very low learning rate for fine-tuning
fe_model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-5),
                 loss='categorical_crossentropy', metrics=['accuracy'])

h_ft = fe_model.fit(X_train_sub, y_train_sub, epochs=10, batch_size=32,
                     validation_data=(X_test[:2000], y_test_cat[:2000]),
                     callbacks=[early_stop], verbose=1)

ft_score = fe_model.evaluate(X_test, y_test_cat, verbose=0)
print(f'\nFine-Tuned — Test Accuracy: {ft_score[1]:.4f}')

## 5. Train from Scratch vs Transfer Learning

In [ ]:
# Baseline: Train a CNN from scratch
scratch_model = keras.Sequential([
    layers.Conv2D(32, 3, activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu'),
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])
scratch_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
h_scratch = scratch_model.fit(X_train_sub, y_train_sub, epochs=20, batch_size=32,
                               validation_data=(X_test[:2000], y_test_cat[:2000]),
                               callbacks=[early_stop], verbose=0)
scratch_score = scratch_model.evaluate(X_test, y_test_cat, verbose=0)
print(f'From Scratch: {scratch_score[1]:.4f}')
print(f'Feature Extraction: {fe_score[1]:.4f}')
print(f'Fine-Tuned: {ft_score[1]:.4f}')

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for h, name in [(h_scratch, 'Scratch'), (h_fe, 'Feature Extraction'), (h_ft, 'Fine-Tuned')]:
    axes[0].plot(h.history['val_accuracy'], label=name)
    axes[1].plot(h.history['val_loss'], label=name)

axes[0].set_title('Validation Accuracy'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].set_title('Validation Loss'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle('Transfer Learning vs Training from Scratch', fontsize=14)
plt.tight_layout()
plt.show()

## 6. When to Use Transfer Learning

| Scenario | Strategy | Why |
|----------|----------|-----|
| Small dataset, similar domain | Feature Extraction | Pre-trained features are relevant |
| Small dataset, different domain | Fine-tune top layers | Need to adapt features |
| Large dataset, similar domain | Fine-tune all layers | Enough data to retrain |
| Large dataset, different domain | Train from scratch | Pre-trained features not useful |

### Key Tips
1. Always use a **very low learning rate** for fine-tuning (1e-5 to 1e-4)
2. **Freeze BatchNorm** layers when fine-tuning (`training=False`)
3. Start with **feature extraction**, then fine-tune if needed
4. Use **MobileNetV2** or **EfficientNet** for resource-constrained environments
5. **Data augmentation** is critical with small datasets